# Проект 1. Шеринг и отток подписчиков ПРАЙМ

## Бизнес-вопрос

Связан ли активный шеринг подписки с уровнем оттока и различается ли эта связь между тарифами до и после изменения цены с 1 марта 2026 года?

## Гипотезы

**Гипотеза 1.** Shared-подписки имеют более низкий churn, чем solo-подписки.

**Гипотеза 2.** После изменения цены с 1 марта разница в churn между shared- и solo-подписками увеличивается.

## Периметр анализа

Анализ проводится на моём персональном сегменте клиентов, заданном через `prime.assignments.client_filter`. В анализ включаются подписки, владельцами которых являются клиенты этого сегмента.

## Зерно данных

Одна строка основной аналитической витрины — одна подписка, активная на начало конкретного месяца.

Статус подписки определяется на начало каждого месяца:
- **shared** — есть хотя бы один активный дополнительный участник;
- **solo** — активных дополнительных участников нет.

## Период

Январь–июль 2026 года.

Август исключён из основного анализа, поскольку данные за месяц неполные.

Для анализа изменения после 1 марта используются два периода:
- **до 1 марта** — январь–февраль 2026 года;
- **после 1 марта** — март–июль 2026 года.

## Основная метрика

Месячный churn:

**число подписок, завершившихся в течение месяца / число подписок, активных на начало месяца.**

Основной анализ изменения churn проводится на non-trial подписках. Trial-подписки предварительно проверяются отдельно.

## 1. Подготовка данных

### Подключение к базе


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sqlalchemy import text
from prime.config import get_engine

engine = get_engine()


def q(sql: str, **params) -> pd.DataFrame:
    """Выполнить SQL-запрос и вернуть DataFrame."""
    try:
        with engine.connect() as conn:
            conn.execute(text("set time zone 'UTC'"))
            return pd.read_sql(text(sql), conn, params=params)
    finally:
        engine.dispose()


print("подключились как:", q("select current_user as login").iloc[0, 0])

### Получение персонального сегмента

In [ ]:
CLIENT_FILTER = q(
    "select client_filter from prime.assignments where login = current_user"
).iloc[0, 0]

print(CLIENT_FILTER)

### Загрузка исходных данных

In [ ]:
SEGMENT = f"""
    select client_id
    from prime.clients
    where {CLIENT_FILTER}
"""

clients = q(f"""
    select
        client_id,
        registered_at,
        region_code,
        acquisition_channel
    from prime.clients
    where {CLIENT_FILTER}
    order by client_id
""")

subs = q(f"""
    select
        subscription_id,
        owner_client_id,
        tariff_id,
        started_at,
        ended_at,
        is_trial,
        cancel_reason
    from prime.subscriptions
    where owner_client_id in ({SEGMENT})
    order by subscription_id
""")

sub_members = q(f"""
    select
        m.subscription_id,
        m.client_id as member_id,
        m.joined_at,
        m.left_at
    from prime.subscription_members m
    join prime.subscriptions s
        on s.subscription_id = m.subscription_id
    where s.owner_client_id in ({SEGMENT})
    order by m.subscription_id, m.client_id
""")

tariffs = q("""
    select *
    from prime.tariffs
    order by tariff_id, valid_from
""")

### Проверка объёма и зерна исходных данных

In [ ]:
print("Клиентов в моём сегменте:", len(clients))
print("Уникальных клиентов:", clients["client_id"].nunique())

print()
print("Подписок моих клиентов:", len(subs))
print("Уникальных подписок:", subs["subscription_id"].nunique())

print()
print("Строк в subscription_members:", len(sub_members))
print("Подписок, которыми хотя бы раз делились:",
      sub_members["subscription_id"].nunique())

### Проверка уникальности ключей

In [ ]:
assert clients["client_id"].is_unique
assert subs["subscription_id"].is_unique

print("Проверки уникальности пройдены")

### Проверка временного покрытия данных

In [ ]:
print("Подписки:")
print("минимальная дата старта:", subs["started_at"].min())
print("максимальная дата старта:", subs["started_at"].max())
print("максимальная дата окончания:", subs["ended_at"].max())

print()
print("Участники:")
print("минимальная дата подключения:", sub_members["joined_at"].min())
print("максимальная дата подключения:", sub_members["joined_at"].max())
print("максимальная дата выхода:", sub_members["left_at"].max())

### Проверка качества дат и trial-подписок

In [ ]:
print("Подписок с окончанием раньше старта:",
      (subs["ended_at"] < subs["started_at"]).sum())

print("Участников с выходом раньше подключения:",
      (sub_members["left_at"] < sub_members["joined_at"]).sum())

print("Trial-подписок:")
print(subs["is_trial"].value_counts(dropna=False))

## 2. Расчёт метрик и проверка гипотез

### Период анализа
*учитываются только полные месяцы*

In [ ]:
months = pd.DataFrame({
    "month_start": pd.date_range(
        start="2026-01-01",
        end="2026-07-01",
        freq="MS",
        tz="UTC"
    )
})

months["next_month_start"] = (
    months["month_start"] + pd.offsets.MonthBegin(1)
)

months

### Формирование витрины subscription × month
*все возможные комбинации подписки и месяца*

In [ ]:
sub_month = subs.merge(
    months,
    how="cross"
)

print("Строк до фильтра активных:", len(sub_month))

### Отбор подписок, активных на начало месяца

In [ ]:
active_start = (
    (sub_month["started_at"] <= sub_month["month_start"])
    &
    (
        sub_month["ended_at"].isna()
        | (sub_month["ended_at"] > sub_month["month_start"])
    )
)

mart = sub_month[active_start].copy()

print("Активных subscription-month:", len(mart))

### Расчёт churn

In [ ]:
mart["churned"] = (
    mart["ended_at"].notna()
    & (mart["ended_at"] >= mart["month_start"])
    & (mart["ended_at"] < mart["next_month_start"])
)
duplicates = mart.duplicated(
    subset=["subscription_id", "month_start"]
).sum()

print("Дублей subscription_id × month:", duplicates)

assert duplicates == 0

### Определение активных участников на начало месяца

In [ ]:
member_month = sub_members.merge(
    months,
    how="cross"
)
member_active = (
    (member_month["joined_at"] <= member_month["month_start"])
    &
    (
        member_month["left_at"].isna()
        | (member_month["left_at"] > member_month["month_start"])
    )
)

member_month = member_month[member_active].copy()
member_count = (
    member_month
    .groupby(
        ["subscription_id", "month_start"],
        as_index=False
    )
    .agg(
        n_active_members=("member_id", "nunique")
    )
)

member_count.head()

### Определение shared / solo

Для каждой подписки определяется наличие активных дополнительных участников на начало месяца. Участники предварительно агрегируются до зерна «подписка × месяц»

In [ ]:
mart = mart.drop(
    columns=[
        "n_active_members",
        "n_active_members_x",
        "n_active_members_y",
        "shared"
    ],
    errors="ignore"
)

rows_before = len(mart)

mart = mart.merge(
    member_count,
    on=["subscription_id", "month_start"],
    how="left",
    validate="1:1"
)

mart["n_active_members"] = (
    mart["n_active_members"]
    .fillna(0)
    .astype(int)
)

mart["shared"] = mart["n_active_members"] > 0

assert len(mart) == rows_before
assert mart[
    ["subscription_id", "month_start"]
].duplicated().sum() == 0

print("Зерно сохранено")

### Проверка гипотезы 1. Общий churn: shared vs solo
Проверяется исходная гипотеза без дополнительной сегментации.

In [ ]:
h1 = (
    mart
    .groupby("shared", as_index=False)
    .agg(
        active=("subscription_id", "size"),
        churned=("churned", "sum")
    )
)

h1["churn_pct"] = (
    h1["churned"]
    / h1["active"]
    * 100
)

h1.round(2)

### Добавление тарифного сегмента

Проверяется однозначность соответствия tariff_id названию тарифа. Уникальный справочник тарифов присоединяется к аналитической витрине с контролем кратности и сохранения зерна.

In [ ]:
tariff_check = (
    tariffs
    .groupby("tariff_id", as_index=False)
    .agg(
        rows=("tariff_id", "size"),
        tariff_names=("tariff_name", "nunique")
    )
)

assert (tariff_check["tariff_names"] == 1).all()

tariff_lookup = (
    tariffs[
        [
            "tariff_id",
            "tariff_name",
            "share_slots"
        ]
    ]
    .drop_duplicates()
)

assert tariff_lookup["tariff_id"].is_unique

print(tariff_lookup)
print("Уникальных tariff_id:", tariff_lookup["tariff_id"].nunique())
print("Строк в справочнике:", len(tariff_lookup))


# Удаляем тарифные поля при повторном запуске ячейки
mart = mart.drop(
    columns=[
        "tariff_name",
        "tariff_name_x",
        "tariff_name_y",
        "share_slots",
        "share_slots_x",
        "share_slots_y"
    ],
    errors="ignore"
)

rows_before = len(mart)

mart = mart.merge(
    tariff_lookup,
    on="tariff_id",
    how="left",
    validate="m:1"
)

assert len(mart) == rows_before
assert mart["tariff_name"].notna().all()
assert mart[
    ["subscription_id", "month_start"]
].duplicated().sum() == 0

print("Тариф добавлен, зерно сохранено")

### Результат Гипотезы 1 и уточнение анализа

H1 в агрегированном срезе не подтверждается: churn shared- и solo-подписок практически совпадает.

Для оценки неоднородности связи показатель рассматривается отдельно по тарифам.

**Уточнённая H1:** связь между шерингом и churn различается в зависимости от тарифа.

### H1 в разрезе тарифов

In [ ]:
h1_tariff = (
    mart
    .groupby(
        ["tariff_name", "shared"],
        as_index=False
    )
    .agg(
        active=("subscription_id", "size"),
        churned=("churned", "sum")
    )
)

h1_tariff["churn_pct"] = (
    h1_tariff["churned"]
    / h1_tariff["active"]
    * 100
)

h1_tariff.round(2)

### Проверка trial-подписок

Trial-подписки проверяются отдельно, так как их поведение может отличаться от платных подписок.

In [ ]:
trial_tariff_check = (
    mart
    .groupby(
        ["tariff_name", "is_trial", "shared"],
        as_index=False
    )
    .agg(
        active=("subscription_id", "size"),
        churned=("churned", "sum")
    )
)

trial_tariff_check["churn_pct"] = (
    trial_tariff_check["churned"]
    / trial_tariff_check["active"]
    * 100
)

trial_tariff_check.round(2)

### Уточнённая Гипотеза 2: После 1 марта shared-подписки Расширенного тарифа более устойчивы относительно solo-подписок.

### Расчет для гипотезы 2. Изменение churn после 1 марта
Для анализа изменения после 1 марта используются non-trial подписки. Сравнение проводится отдельно по тарифу и наличию активного шеринга.

In [ ]:
paid = mart[~mart["is_trial"]].copy()
paid["period"] = np.where(
    paid["month_start"] < pd.Timestamp("2026-03-01", tz="UTC"),
    "до 1 марта",
    "после 1 марта"
)
h2 = (
    paid
    .groupby(
        ["tariff_name", "period", "shared"],
        as_index=False
    )
    .agg(
        active=("subscription_id", "size"),
        churned=("churned", "sum")
    )
)

h2["churn_pct"] = (
    h2["churned"]
    / h2["active"]
    * 100
)

h2.round(2)

In [ ]:
h2_pivot = (
    h2
    .pivot(
        index=["tariff_name", "shared"],
        columns="period",
        values="churn_pct"
    )
    .reset_index()
)

h2_pivot["изменение_пп"] = (
    h2_pivot["после 1 марта"]
    - h2_pivot["до 1 марта"]
)

h2_pivot.round(2)

### Оценка разницы shared  и solo

In [ ]:
gap = (
    h2
    .pivot_table(
        index=["tariff_name", "period"],
        columns="shared",
        values="churn_pct"
    )
    .reset_index()
    .rename(
        columns={
            False: "solo_churn",
            True: "shared_churn"
        }
    )
)

gap["shared_minus_solo_pp"] = (
    gap["shared_churn"]
    - gap["solo_churn"]
)

gap.round(2)

### Оценка изменения разницы shared и solo после 1 марта

In [ ]:
gap_change = (
    gap
    .pivot(
        index="tariff_name",
        columns="period",
        values="shared_minus_solo_pp"
    )
    .reset_index()
)

gap_change["изменение_разрыва_пп"] = (
    gap_change["после 1 марта"]
    - gap_change["до 1 марта"]
)

gap_change.round(2)

### Помесячная динамика

Помесячный анализ используется для оценки устойчивости выявленных различий между сегментами.

In [ ]:
monthly_segments = (
    paid
    .groupby(
        ["month_start", "tariff_name", "shared"],
        as_index=False
    )
    .agg(
        active=("subscription_id", "size"),
        churned=("churned", "sum")
    )
)

monthly_segments["churn_pct"] = (
    monthly_segments["churned"]
    / monthly_segments["active"]
    * 100
)

monthly_segments["segment"] = (
    monthly_segments["tariff_name"]
    + " | "
    + np.where(
        monthly_segments["shared"],
        "shared",
        "solo"
    )
)


monthly_segments_display = monthly_segments[
    [
        "month_start",
        "segment",
        "active",
        "churned",
        "churn_pct"
    ]
].copy()

monthly_segments_display["month_start"] = (
    monthly_segments_display["month_start"]
    .dt.strftime("%Y-%m")
)

monthly_segments_display["churn_pct"] = (
    monthly_segments_display["churn_pct"].round(2)
)

monthly_segments_display

In [ ]:
plot_data = (
    monthly_segments
    .pivot(
        index="month_start",
        columns="segment",
        values="churn_pct"
    )
)

ax = plot_data.plot(
    figsize=(10, 5),
    marker="o"
)

ax.axvline(
    pd.Timestamp("2026-03-01", tz="UTC"),
    linestyle="--"
)

ax.set_title("Динамика churn по тарифу и наличию шеринга")
ax.set_xlabel("Месяц")
ax.set_ylabel("Churn, %")
ax.legend(title="Сегмент")

plt.tight_layout()
plt.show()

### Результат проверки гипотезы 2

Наиболее заметное изменение выявлено в solo-сегменте Расширенного тарифа. После 1 марта его churn увеличивается, тогда как churn shared-подписок в среднем остаётся стабильным.

Различие не является устойчивым во всех месяцах. H2 подтверждается частично.

## 3. Факторная декомпозиция и выводы

Изменение общего churn раскладывается на эффект изменения структуры сегментов и эффект изменения churn внутри сегментов.

In [ ]:
decomp = (
    h2
    .pivot(
        index=["tariff_name", "shared"],
        columns="period",
        values=["active", "churned"]
    )
)

decomp.columns = [
    "_".join(col)
    for col in decomp.columns
]

decomp = decomp.reset_index()

decomp

### Расчет весов сегментов и churn

In [ ]:
total_active_before = decomp["active_до 1 марта"].sum()
total_active_after = decomp["active_после 1 марта"].sum()

decomp["weight_before"] = (
    decomp["active_до 1 марта"]
    / total_active_before
)

decomp["weight_after"] = (
    decomp["active_после 1 марта"]
    / total_active_after
)

decomp["churn_before"] = (
    decomp["churned_до 1 марта"]
    / decomp["active_до 1 марта"]
)

decomp["churn_after"] = (
    decomp["churned_после 1 марта"]
    / decomp["active_после 1 марта"]
)

### Факторная декомпозиция

In [ ]:
decomp["mix_effect"] = (
    (decomp["weight_after"] - decomp["weight_before"])
    * decomp["churn_before"]
)

decomp["rate_effect"] = (
    decomp["weight_after"]
    * (decomp["churn_after"] - decomp["churn_before"])
)
decomp["mix_effect_pp"] = decomp["mix_effect"] * 100
decomp["rate_effect_pp"] = decomp["rate_effect"] * 100

decomp[
    [
        "tariff_name",
        "shared",
        "weight_before",
        "weight_after",
        "churn_before",
        "churn_after",
        "mix_effect_pp",
        "rate_effect_pp"
    ]
].round(4)

### Контроль декомпозиции
Суммарный вклад факторов сопоставляется с фактическим изменением общего churn.

### Расчет общего изменения churn после первого марта

In [ ]:
churn_before_total = (
    decomp["churned_до 1 марта"].sum()
    / decomp["active_до 1 марта"].sum()
)

churn_after_total = (
    decomp["churned_после 1 марта"].sum()
    / decomp["active_после 1 марта"].sum()
)

delta_actual_pp = (
    churn_after_total - churn_before_total
) * 100

delta_decomp_pp = (
    decomp["mix_effect_pp"].sum()
    + decomp["rate_effect_pp"].sum()
)

print("Churn до:", round(churn_before_total * 100, 4), "%")
print("Churn после:", round(churn_after_total * 100, 4), "%")
print("Фактическое изменение:", round(delta_actual_pp, 4), "п.п.")
print("Сумма факторов:", round(delta_decomp_pp, 4), "п.п.")

assert np.isclose(
    delta_actual_pp,
    delta_decomp_pp
)

print("Контроль декомпозиции пройден")

### Расчет churn до и после 1 марта по сегментам

In [ ]:
plot_h2 = h2.copy()

plot_h2["segment"] = (
    plot_h2["tariff_name"]
    + " — "
    + np.where(
        plot_h2["shared"],
        "shared",
        "solo"
    )
)

plot_h2 = (
    plot_h2
    .pivot(
        index="segment",
        columns="period",
        values="churn_pct"
    )
)

ax = plot_h2[
    ["до 1 марта", "после 1 марта"]
].plot(
    kind="bar",
    figsize=(9, 5)
)

ax.set_title("Churn до и после 1 марта по сегментам")
ax.set_xlabel("Сегмент")
ax.set_ylabel("Churn, %")
ax.legend(title="Период")

plt.xticks(rotation=20, ha="right")
plt.tight_layout()
plt.show()

### Расчет факторов изменения churn после 1 марта

In [ ]:
decomp_plot = decomp.copy()

decomp_plot["segment"] = (
    decomp_plot["tariff_name"]
    + " | "
    + np.where(
        decomp_plot["shared"],
        "shared",
        "solo"
    )
)

effects = pd.DataFrame({
    "factor": [
        "Изменение структуры",
        "Базовый | solo",
        "Базовый | shared",
        "Расширенный | solo",
        "Расширенный | shared"
    ],
    "effect_pp": [
        decomp_plot["mix_effect_pp"].sum(),
        decomp_plot.loc[
            (decomp_plot["tariff_name"] == "Базовый")
            & (~decomp_plot["shared"]),
            "rate_effect_pp"
        ].iloc[0],
        decomp_plot.loc[
            (decomp_plot["tariff_name"] == "Базовый")
            & (decomp_plot["shared"]),
            "rate_effect_pp"
        ].iloc[0],
        decomp_plot.loc[
            (decomp_plot["tariff_name"] == "Расширенный")
            & (~decomp_plot["shared"]),
            "rate_effect_pp"
        ].iloc[0],
        decomp_plot.loc[
            (decomp_plot["tariff_name"] == "Расширенный")
            & (decomp_plot["shared"]),
            "rate_effect_pp"
        ].iloc[0],
    ]
})

effects.round(4)
ax = effects.plot(
    x="factor",
    y="effect_pp",
    kind="bar",
    legend=False,
    figsize=(9, 5)
)

ax.axhline(0, linewidth=1)

ax.set_title("Вклад факторов в изменение churn после 1 марта")
ax.set_xlabel("Фактор")
ax.set_ylabel("Вклад в изменение churn, п.п.")

plt.xticks(rotation=20, ha="right")
plt.tight_layout()
plt.show()

### Итог по уточнённым гипотезам

**Уточнённая H1: связь между шерингом и churn различается в зависимости от тарифа — подтверждается.**  
Для Базового и Расширенного тарифов направление различий между shared- и solo-подписками не совпадает. На Базовом тарифе churn shared-подписок выше, тогда как на Расширенном — ниже.

**Уточнённая H2: после 1 марта shared-подписки Расширенного тарифа более устойчивы относительно solo-подписок — подтверждается частично.**  
В Расширенном тарифе churn solo-подписок увеличился с **3,69% до 3,87%**, тогда как churn shared-подписок остался на уровне **3,74%**. Разрыв между shared и solo изменился с **+0,04 п.п. до −0,13 п.п.**, то есть примерно на **−0,18 п.п.** в пользу shared. При этом помесячная динамика показывает, что различие не является устойчивым во всех месяцах.

**Дополнительная сегментация показала, что связь между шерингом и churn зависит от тарифа. Наиболее заметное различие после 1 марта наблюдается в Расширенном тарифе: churn solo-подписок увеличивается, тогда как churn shared-подписок в среднем остаётся стабильным.**

## Итог

**Гипотеза 1 не подтверждается в общем виде.** Churn shared- и solo-подписок практически одинаков. При детализации по тарифам направление различий не совпадает.

**Гипотеза 2 подтверждается частично.** После 1 марта в Расширенном тарифе churn solo-подписок увеличивается, тогда как churn shared-подписок в среднем остаётся стабильным. Помесячная динамика показывает отсутствие устойчивого различия во всех месяцах.

Общий churn практически не изменился: 3,7517% до 1 марта и 3,7537% после (+0,0020 п.п.). Факторная декомпозиция показывает, что снижение churn в Базовом solo-сегменте практически компенсировало рост churn в Расширенном solo-сегменте.

Результаты отражают наблюдаемую связь и не подтверждают причинное влияние шеринга на churn.

## Бизнес-рекомендация

Приоритетным сегментом для дополнительного исследования являются solo-подписки Расширенного тарифа, в которых после 1 марта наблюдается рост churn.

Полученные результаты не подтверждают причинное влияние шеринга на удержание. Для проверки эффекта целесообразно провести A/B-тест механики вовлечения владельцев solo-подписок Расширенного тарифа в шеринг.

## Ограничения

- Анализ выявляет наблюдаемую связь, но не устанавливает причинность.
- Период до 1 марта включает только январь–февраль 2026 года.
- Различие по Расширенному тарифу не является устойчивым во всех месяцах.
- Август исключён из-за неполноты данных.